# Review: Namespace & Scope로 다시 보는 Class 1-6

Class 1-6의 예제를 이름과 객체의 관계로 다시 살펴본다. Namespace, Scope, Binding을 구분하고, 조건문·반복문·Generator에서 이름과 상태가 어떻게 유지되는지 확인한다.


## 용어 정리

### 1. Namespace (네임스페이스)

정의: 이름(식별자)과 객체를 매핑하는 공간

```python
# Namespace는 딕셔너리처럼 동작

{
    'x': <int object at 0x...>,
    'name': <str object at 0x...>,
    'func': <function object at 0x...>
}
```

**실제 확인**: `globals()`, `locals()` 딕셔너리


In [ ]:
# Namespace 확인
print("=== Namespace 탐색 ===")

x = 10
name = "Alice"

# 전역 Namespace
print("\n전역 Namespace에서 찾기:")
print(f"'x' in globals(): {'x' in globals()}")
print(f"globals()['x']: {globals()['x']}")

# 함수 내 Namespace


def show_namespace():
  y = 20
  print("\n지역 Namespace:")
  print(f"locals(): {locals()}")
  print(f"'y' in locals(): {'y' in locals()}")
  print(f"'x' in locals(): {'x' in locals()}")


show_namespace()

=== Namespace 탐색 ===

전역 Namespace에서 찾기:
'x' in globals(): True
globals()['x']: 10

지역 Namespace:
locals(): {'y': 20}
'y' in locals(): True
'x' in locals(): False


### 2. Scope (스코프)

정의: 변수(이름)를 참조할 수 있는 코드 범위

**LEGB 룰**:

```
L (Local)      : 함수 내부
E (Enclosing)  : 중첩 함수의 외부 함수
G (Global)     : 모듈 레벨
B (Built-in)   : Python 내장
```


In [ ]:
# LEGB 룰 확인
print("=== LEGB 룰 ===")

x = 'Global x'


def outer():
  x = 'Enclosing x'

  def inner():
    x = 'Local x'
    print(f"1. Local: {x}")

  inner()
  print(f"2. Enclosing: {x}")


outer()
print(f"3. Global: {x}")
print(f"4. Built-in: {len}")

### 3. Binding (바인딩)

정의: 이름과 값(객체)을 연결하는 행위

```python
x = 10  # Binding: 'x'라는 이름을 10이라는 객체에 연결
```

**바인딩이 일어나는 경우**:

1. 할당문: `x = 10`
2. 함수 정의: `def func():`
3. 클래스 정의: `class MyClass:`
4. import: `import math`
5. for 문: `for i in range(10):`


In [ ]:
# Binding 확인
print("=== Binding ===")

# 1. 할당문으로 Binding
x = 10
print(f"x가 가리키는 객체 id: {id(x)}")

# 2. 재할당 = 새로운 Binding
old_id = id(x)
x = 20
print(f"재할당 후 id: {id(x)}")
print(f"id 변경됨? {old_id != id(x)}")

# 3. 여러 이름이 같은 객체에 Binding
y = x
print(f"\nx와 y가 같은 객체? {x is y}")
print(f"x id: {id(x)}, y id: {id(y)}")

### 4. Lifetime (생존 기간)

정의: 객체가 메모리에 존재하는 기간

```python
def func():
    x = 10  # 함수 시작: x 생성
    return x
# 함수 종료: x 소멸

```


In [ ]:
# Lifetime 확인
import sys

print("=== Lifetime ===")


def create_temp():
  temp = [1, 2, 3]
  print(f"함수 내부 - temp id: {id(temp)}")
  print(f"함수 내부 - refcount: {sys.getrefcount(temp) - 1}")
  return temp


result = create_temp()
print(f"\n함수 외부 - result id: {id(result)}")
print(f"함수 외부 - refcount: {sys.getrefcount(result) - 1}")
print("\n→ 같은 객체지만, 'temp' 이름은 소멸됨")

### 5. Visibility (가시성)

정의: 이름에 접근 가능한지 여부

```python
_private_var  # 외부에서 접근 비추천 (관례)
__name__      # 특수 메서드/속성
public_var    # 외부에서 자유롭게 접근
```


In [ ]:
# Visibility
print("=== Visibility ===")


class Example:
  def __init__(self):
    self.public = "누구나 접근"
    self._protected = "관례상 내부용"
    self.__private = "이름 맹글링"


obj = Example()
print(f"public: {obj.public}")
print(f"_protected: {obj._protected}")
# print(f"__private: {obj.__private}")  # AttributeError!
print(f"__private (맹글링): {obj._Example__private}")

### 6. Context (컨텍스트)

정의: 실행 환경 또는 현재 활성 Scope의 상태

```python
with open('file.txt') as f:
    # f는 이 컨텍스트 내에서만 유효
    content = f.read()
# 컨텍스트 종료: f는 자동으로 닫힘

```


In [ ]:
# Context 시뮬레이션
print("=== Context ===")


class FileSimulator:
  def __init__(self, name):
    self.name = name

  def __enter__(self):
    print(f"'{self.name}' 컨텍스트 진입")
    return self

  def __exit__(self, *args):
    print(f"'{self.name}' 컨텍스트 종료")

  def read(self):
    return "file content"


print("컨텍스트 전")
with FileSimulator('test.txt') as f:
  print(f"컨텍스트 내부: {f.read()}")
print("컨텍스트 후")
# f는 여전히 존재하지만, 파일은 닫힘

## Class 1-2 복습: Token과 Namespace

Token 중 Identifier는 이름을 나타낸다. Token을 구분하는 일과, 그 이름이 어느 Scope의 객체를 가리키는지 결정하는 일은 구분해서 보자.


In [ ]:
# Class 1-2 재해석
import tokenize
import io

print("=== Token과 Namespace ===")

code = "x = 10 + y"
tokens = tokenize.generate_tokens(io.StringIO(code).readline)

print(f"코드: {code}")
print("\nToken 분석:")
for token in tokens:
  if token.type == tokenize.NAME:
    print(f"  NAME '{token.string}' → Namespace에서 찾을 이름")
  elif token.type == tokenize.NUMBER:
    print(f"  NUMBER '{token.string}' → 직접적인 객체 생성")
  elif token.type == tokenize.OP:
    print(f"  OP '{token.string}' → 연산 수행")

### AST와 Scope

같은 이름도 코드에서 정의하고 사용하는 위치에 따라 다르게 해석된다. 아래 함수 정의와 호출 예제에서 이름을 연결하는 시점과 찾는 시점을 구분해 보자.


In [ ]:
# AST와 Scope
import ast

print("=== AST와 Scope ===")

code = """
x = 10
def func():
    y = 20
"""

tree = ast.parse(code)
print("AST 구조:")
print(ast.dump(tree, indent=2))

print("\n해석:")
print("  x = 10        → Global Namespace에 바인딩")
print("  def func():   → Global Namespace에 'func' 바인딩")
print("  y = 20        → func의 Local Namespace에 바인딩")

## Class 3 복습: Expression과 Name Resolution

Expression에 이름이 들어 있으면 해당 이름이 가리키는 객체를 찾아 평가한다. 중첩 함수 예제에서 LEGB 순서에 따라 선택되는 값을 추적해 보자.


In [ ]:
# Class 3 재해석: Expression과 Namespace
print("=== Expression 평가와 LEGB ===")

x = 100  # Global


def outer():
  x = 50  # Enclosing

  def inner():
    # x를 Expression에서 사용
    result = x + 10  # 어떤 x?
    print(f"inner - result = {result}")
    print("  → Local에 x 없음")
    print("  → Enclosing에서 x 발견: 50")
    print("  → 50 + 10 = 60")

  inner()


outer()

print("\nLEGB 순서로 x를 찾음:")
print("  L (Local) → 없음")
print("  E (Enclosing) → 50 발견! ✅")

### eval()과 Namespace

**eval()은 특정 Namespace에서 Expression을 평가**


In [ ]:
# eval과 Namespace
print("=== eval과 Namespace ===")

x = 100

# 기본: 현재 Namespace 사용
result = eval('x + 10')
print(f"eval('x + 10') = {result}")

# 사용자 정의 Namespace
custom_namespace = {'x': 50, 'y': 20}
result = eval('x + y', custom_namespace)
print(f"\ncustom namespace:")
print(f"  eval('x + y', {{'x': 50, 'y': 20}}) = {result}")
print("  → 지정한 Namespace에서 이름을 찾음")

## Class 4 복습: Object와 Binding

변수는 객체 자체가 아니라 객체를 가리키는 이름이다. 다른 이름을 같은 객체에 연결하는 것과, 이름을 새 객체에 연결하는 것을 비교한다.


In [ ]:
# Class 4 재해석: Binding의 본질
print("=== Binding의 본질 ===")

# 1. Binding 생성
x = [1, 2, 3]
print(f"1. x = [1, 2, 3]")
print(f"   Namespace에 'x' → <list object> 추가")
print(f"   id: {id(x)}")

# 2. 새로운 이름으로 같은 객체 Binding
y = x
print(f"\n2. y = x")
print(f"   Namespace에 'y' → 같은 <list object> 추가")
print(f"   y id: {id(y)}")
print(f"   같은 객체? {x is y}")

# 3. 한 이름 수정 → 같은 객체 수정
x.append(4)
print(f"\n3. x.append(4)")
print(f"   x: {x}")
print(f"   y: {y}")
print(f"   → 같은 객체를 가리키므로 둘 다 변경")

# 4. Rebinding = 다른 객체 가리키기
x = [5, 6, 7]
print(f"\n4. x = [5, 6, 7] (Rebinding)")
print(f"   x: {x}, id: {id(x)}")
print(f"   y: {y}, id: {id(y)}")
print(f"   → x는 이제 다른 객체를 가리킴")

### Namespace는 계층적이다

**각 Scope마다 독립적인 Namespace**


In [ ]:
# Namespace 계층
print("=== Namespace 계층 ===")

x = 'global'


def func():
  x = 'local'
  print(f"func 내부:")
  print(f"  locals(): {locals()}")
  print(f"  'x' in locals(): True")
  print(f"  x = {x}")


print("Global Namespace:")
print(f"  x = {x}")
print()

func()

print(f"\nfunc 종료 후:")
print(f"  x = {x}  (여전히 'global')")
print("  → 함수의 Local Namespace는 독립적")

## Class 5 복습: 조건문과 Scope

`if` 블록은 함수처럼 별도의 Local Scope를 만들지 않는다. 아래 예제에서 블록 안에 할당한 이름을 블록 뒤에서도 사용할 수 있는지 확인해 보자.


In [ ]:
# Class 5 재해석: 조건문은 Scope를 만들지 않음
print("=== 조건문과 Scope ===")


def test_scope():
  if True:
    x = 10  # 블록 내에서 Binding
    print(f"if 블록 내: x = {x}")

  # Python: if 블록 밖에서도 x 접근 가능!
  print(f"if 블록 밖: x = {x}")
  print("\n→ Python의 if/for는 새로운 Scope를 만들지 않음")
  print("→ 함수만 새로운 Scope 생성")


test_scope()

# C/Java와 비교
print("\nC/Java:")
print("""
if (true) {
    int x = 10;
}
// x는 여기서 접근 불가 ❌
""")

### for 문의 Loop Variable Binding

`for` 문도 별도의 Scope를 만들지 않는다. 반복이 끝난 뒤 반복 변수에 어떤 값이 남는지 확인해 보자.


In [ ]:
# for 문과 Binding
print("=== for 문과 Binding ===")


def test_for():
  for i in range(3):
    print(f"loop: i = {i}")

  # Python: loop 밖에서도 i 접근 가능
  print(f"\nloop 후: i = {i}")
  print("→ i는 함수의 Local Namespace에 Binding됨")


test_for()

print("\n주의: 이는 버그의 원인이 될 수 있음!")

## Class 6 복습: Iterator와 Namespace

Generator는 `yield`로 실행을 멈춘 동안에도 지역 변수와 실행 위치를 유지한다. 다음 `next()` 호출에서 어디부터 이어지는지 확인한다.


In [ ]:
# Class 6 재해석: Generator와 Namespace
print("=== Generator와 Namespace ===")


def counter():
  count = 0  # Generator의 Local Namespace
  while True:
    count += 1
    yield count


# Generator 객체 생성
c1 = counter()
c2 = counter()

print("c1:", next(c1), next(c1), next(c1))
print("c2:", next(c2), next(c2))

print("\n해석:")
print("  각 Generator는 독립적인 Namespace 유지")
print("  count는 각 Generator 내에서만 존재")
print("  yield 사이에 상태(Namespace) 보존")

### Comprehension의 Scope

**Python 3: Comprehension은 자체 Scope 보유**


In [ ]:
# Comprehension의 Scope
print("=== Comprehension의 Scope ===")

x = 'global'

# List comprehension의 변수는 외부로 유출되지 않음
result = [x for x in range(3)]
print(f"result: {result}")
print(f"외부 x: {x}")
print("→ Comprehension의 x는 독립적인 Scope")

# for 문과 비교
print("\nfor 문:")
for i in range(3):
  pass
print(f"외부에서 i 접근: {i}")
print("→ for 문의 i는 외부로 유출")

print("\nPython 2에서는 Comprehension도 유출됨 (주의!)")

## Namespace와 이름 찾기

함수 안에서 이름을 찾는 기본 순서는 다음과 같다.

```text
Local → Enclosing → Global → Built-in
```

Namespace가 물리적으로 서로 안에 들어 있는 것은 아니다. 이 순서는 코드가 이름을 해석할 때 따르는 검색 순서다.


In [ ]:
# 종합 예제: 모든 개념 통합
print("=== 종합 예제 ===")

# Global Namespace
counter = 0


def outer(x):
  # Enclosing Namespace
  multiplier = 2

  def inner(y):
    # Local Namespace
    global counter
    counter += 1

    # Name Resolution: LEGB
    result = x * multiplier + y  # x(param), multiplier(enclosing), y(param)

    # Built-in
    return len(str(result))  # len은 Built-in

  return inner


# 실행
func = outer(10)
print(f"결과: {func(5)}")
print(f"counter: {counter}")

print("\n분석:")
print("  x=10      → outer의 Parameter (Enclosing)")
print("  multiplier → Enclosing Namespace")
print("  y=5       → inner의 Parameter (Local)")
print("  counter   → Global Namespace (global 선언)")
print("  len       → Built-in Namespace")
print("  str       → Built-in Namespace")

## Class 7에서 이어서 다룰 내용

- 매개변수에 인자를 연결하는 과정과 `*args`, `**kwargs`
- `global`, `nonlocal`을 사용한 이름의 재할당
- Closure가 바깥 함수의 변수를 유지하는 방식
- 함수 호출에 따른 Stack Frame과 Call Stack의 변화


In [ ]:
# Class 7 미리보기
print("=== Class 7 미리보기 ===")

# 1. Closure = Enclosing Namespace 포획


def make_multiplier(n):
  def multiplier(x):
    return x * n  # n은 Enclosing에서 포획
  return multiplier


times2 = make_multiplier(2)
times3 = make_multiplier(3)

print(f"times2(5) = {times2(5)}")
print(f"times3(5) = {times3(5)}")
print("→ 각 함수는 자신만의 'n'을 기억")

# 2. nonlocal로 Enclosing 수정


def counter():
  count = 0

  def increment():
    nonlocal count  # Enclosing의 count 수정
    count += 1
    return count

  return increment


c = counter()
print(f"\n{c()}, {c()}, {c()}")
print("→ nonlocal로 Enclosing Namespace 수정")

## 정리와 복습

이름을 할당·정의·Import하면 해당 Namespace에 바인딩이 생긴다. 이름을 다시 할당하는 것과 객체 자체를 수정하는 것은 다르다. 또 이름이 보이는 Scope와 객체가 살아 있는 기간도 구분해야 한다.

1. `globals()`와 `locals()`에서 같은 이름이 서로 다른 객체를 가리키는 예를 찾아보자.
2. `a = b` 뒤에 `a`를 재할당하는 경우와 `a`의 List 내용을 수정하는 경우를 비교해 보자.
3. `if`, `for`, 함수, Comprehension에서 이름의 Scope가 어떻게 다른지 확인해 보자.
4. Generator를 멈췄다가 다시 실행해도 지역 변수의 값이 유지되는 이유를 설명해 보자.
